# Travel Itinerary Assistant: full run


In [1]:
import travel_agent.config  # loads .env and sets the API key
from agents import Runner
from travel_agent.agents.intake_agent import intake_agent
from travel_agent.pipeline import build_plan
from agents import InputGuardrailTripwireTriggered
from agents import SQLiteSession

## 1. Intake: multi-turn chat


In [2]:
session = SQLiteSession("trip-demo-1", "data/sessions.db")

async def say(msg):
    try:
        result = await Runner.run(intake_agent, msg, session=session)
    except InputGuardrailTripwireTriggered as exc:
        print("blocked:", exc.guardrail_result.output.output_info.reason)
        return None
    out = result.final_output
    print("missing:", out.missing_fields)
    print("assistant:", out.next_question)
    return out
out = await say("I want to visit Japan with 2 friends starting October 26 for 7 days, budget 6000 USD")
# out = await say("Write me a poem about my cat")

missing: ['trip_type', 'passport_country', 'origin', 'interests']
assistant: What country is your passport from? Also, what city are you traveling from?


In [3]:
out = await say("I hold a French passport, flying from Miami. Food and temples for vacation.")

missing: []
assistant: None


In [4]:
assert out.is_complete, out.missing_fields
trip = out.trip
print(trip.model_dump_json(indent=2))

{
  "trip_type": "leisure",
  "passport_country": "France",
  "origin": "Miami",
  "destination": "Japan",
  "start_date": "2026-10-26",
  "end_date": "2026-11-01",
  "num_days": 7,
  "travelers": 3,
  "budget_usd": 6000.0,
  "interests": [
    "Food",
    "temples"
  ],
  "meeting_address": null,
  "needs_car_rental": null
}


## 2. Tool smoke test (no LLM)

Confirms mock data files and tools work before running agents.


In [5]:
from travel_agent.tools.transport_tool import search_flights_raw
from travel_agent.tools.stay_tool import search_stays_raw
from travel_agent.tools.visa_tool import check_visa_raw
from travel_agent.airports import resolve_airport

origin, dest = resolve_airport(trip.origin), resolve_airport(trip.destination)
print(origin, dest)
print((await search_flights_raw(origin, dest, trip.start_date, trip.end_date, trip.travelers, force_mock=True))["source"])
print(search_stays_raw(trip.destination, trip.num_days - 1, trip.travelers)["options"][:1])
print(check_visa_raw(trip.passport_country, trip.destination)["source"])

MIA KIX
mock
[{'name': 'Sample Kyoto Guesthouse', 'area': 'Kyoto', 'total_cost_usd': 1020, 'why': 'Sample value option near traditional neighborhoods and temple areas.', 'is_estimate': True}]
orizn_api


## 3. Full pipeline

Visa, Activities, Transport and Stay run in parallel. Then budget, then Planner.


In [6]:
plan = await build_plan(trip)
print("warnings:", plan.warnings)
print(plan.budget.model_dump_json(indent=2))

for a in plan.activities.activities:
    print(a.from_tool, a.category, a.estimated_cost_usd, a.name)

warnings: []
{
  "flights_usd": 2400.0,
  "stay_usd": 1020.0,
  "activities_usd": 0.0,
  "local_transport_usd": 0.0,
  "food_usd": 945.0,
  "total_usd": 4365.0,
  "budget_usd": 6000.0,
  "within_budget": true,
  "remaining_usd": 1635.0
}
True food 30.0 Kyoto Karei Seisakusho Kariru
True food 40.0 Nouvelle Vagne Kyoto
True food 25.0 Bungalow Terramachi
True food 15.0 Marumaru Seimen
True food 20.0 Izashi Tedakichi Soba Sho Denshoji
True temple 0.0 Honnouji Temple
True temple 0.0 Hakusan Shrine
True temple 0.0 Tenmangu Shrine
True temple 0.0 Seiganji Temple
False temple 0.0 Mt. Ryozen Temple
False temple 0.0 Kiyomizu Temple
False temple 0.0 Nanzenji Temple


In [7]:
it = plan.itinerary
print(it.title)
print(it.summary)
print("Visa:", it.visa_note)
print("Budget:", it.budget_note)
for d in it.days:
    print(f"Day {d.day} | {d.date} | {d.city}")
    for item in d.items:
        print(" -", item)

Kyoto Leisure Trip to Japan
This 7-day leisure trip to Kyoto, Japan features a balanced exploration of local temples and dining experiences aligned with your interests in food and temples. Activities are spread throughout the days with cultural visits and restaurant meals. Free time is incorporated after listed activities are used. Your arrival day includes airport transfer from MIA to KIX and your departure day includes check-out and return flight back to Miami. Airport transfers are not priced. The weather in late October to early November is typically mild and suitable for outdoor activities.
Visa: Holders of a French passport do not require a visa to enter Japan and can stay visa-free for up to 90 days. Please verify this information on the official Japanese government website, as entry requirements can change.
Budget: The total estimated cost of $4365.0 is within your budget of $6000.0, leaving $1635.0 remaining. This total includes flights at $2400.0, stay at $1020.0, and food at